# Обучение модели ориентации текста (0° / 180°)

Ноутбук обучает компактную свёрточную сеть, которая по кропу со строкой текста предсказывает `p_180` — вероятность того, что
текст перевёрнут вверх ногами.

**Вход** — датасет из `synth_generator.ipynb`:
```
synth_dataset/
  train/images/*.png, train/labels.csv     image_id, label (1 = повёрнут на 180°), text, font, scene, width, height
  val/images/*.png,   val/labels.csv
```
**Выход** — папка `OUT_DIR` (по умолчанию `orient_model/`):
| Файл | Что это |
|---|---|
| `best.pt` | веса лучшей эпохи, конфигурация модели и температура калибровки |
| `model.onnx` | та же модель для инференса: на входе серые кропы 32×256, на выходе `p_180`; температура и сигмоида встроены |
| `history.csv` | метрики по эпохам |
| `metrics.json` | итоговые метрики на валидации |
| `cache/` | препроцессированные массивы, чтобы не декодировать PNG при повторном запуске |

**Как запускать.** Положите ноутбук рядом с папкой датасета (или укажите путь в `DATA_DIR`) и выполните все ячейки по порядку.
Зависимости ставятся первыми ячейками. Если есть видеокарта NVIDIA, обучение идёт на ней, иначе на CPU.

**Детерминизм.** Сиды фиксированы, порядок батчей задаётся генератором с сидом, на GPU включены детерминированные алгоритмы.
Повторный запуск на той же машине даёт те же веса и метрики.

| Раздел | Содержание |
|---|---|
| 1–2 | Параметры, установка библиотек, импорты |
| 3 | Препроцессинг и загрузка датасета, проверки данных |
| 4 | Модель |
| 5 | Метрики и калибровка, проверки модели перед обучением |
| 6 | Обучение (с прогресс-баром) |
| 7 | Итоговая оценка: общая, по сценам, по размеру кропа |
| 8 | Экспорт в ONNX и скорость |
| 9 | Предсказания для папки с картинками |

## 1. Параметры

| Параметр | Смысл |
|---|---|
| `DATA_DIR` | папка датасета с подпапками `train/` и `val/`. Если по этому пути датасета нет, он ищется в `/kaggle/input` (подключённые на Kaggle датасеты); если не найден и там — ноутбук остановится с подсказкой |
| `OUT_DIR` | куда сохранять модель, метрики и кэш |
| `SEED` | сид обучения: инициализация весов, порядок батчей, dropout |
| `H, W` | размер входа модели: кроп приводится к высоте `H` и вписывается в ширину `W` |
| `MODEL` | ширина сети (`width`: 0.5 / 1.0 / 1.5), расширение блоков (`expand`) и режим головы (`mode`, см. раздел 4) |
| `EPOCHS`, `BATCH`, `LR`, `WEIGHT_DECAY`, `WARMUP_STEPS` | оптимизация: AdamW, линейный разогрев и косинусное затухание шага |
| `DETERMINISTIC` | детерминированные алгоритмы на GPU: повторный запуск даёт те же веса, но обучение медленнее (на GTX 1660 SUPER ~700 против ~1 170 картинок/с) |
| `PREDICT_DIR` | папка с картинками для предсказаний в разделе 9; `None` — пропустить |

In [ ]:
from pathlib import Path
import os

DATA_DIR = Path('synth_dataset')     # если такой папки нет, ноутбук поищет датасет в /kaggle/input (Kaggle)
OUT_DIR = Path('orient_model')
SEED = 0

H, W = 32, 256
MODEL = dict(width=1.0, expand=2, mode='anti')

EPOCHS = 15
BATCH = 256
LR = 3e-3
WEIGHT_DECAY = 0.05
WARMUP_STEPS = 500
DETERMINISTIC = True

PREDICT_DIR = None

## 2. Установка библиотек

Ячейка ставит все библиотеки, которые нужны ноутбуку. Версии закреплены ради воспроизводимости; уже установленные нужные версии
pip пропускает.

| Библиотека | Зачем |
|---|---|
| `torch` | модель и обучение. Ставится сборка с CUDA 12.6, если в системе есть видеокарта NVIDIA (`nvidia-smi`), иначе сборка для CPU |
| `numpy`, `pandas` | массивы и таблицы меток |
| `opencv-python-headless` | чтение картинок и препроцессинг |
| `joblib` | параллельный препроцессинг |
| `onnx`, `onnxruntime` | экспорт модели в ONNX и быстрый инференс на CPU |
| `matplotlib` | графики обучения и проверки данных |
| `tqdm` | прогресс-бары препроцессинга и обучения |

Если pip что-то установил или обновил, перезапустите ядро (*Restart Kernel*) и выполните ноутбук заново.

In [ ]:
import shutil

TORCH_INDEX = 'https://download.pytorch.org/whl/' + ('cu126' if shutil.which('nvidia-smi') else 'cpu')
%pip install -q numpy==2.5.3 pandas==3.0.6 opencv-python-headless==5.0.0.93 joblib==1.6.0 onnx==1.23.0 onnxruntime==1.30.0 matplotlib==3.11.2 tqdm==4.70.1
%pip install -q torch==2.14.0 --index-url {TORCH_INDEX}

### Импорты и окружение

Фиксируются сиды, выбирается устройство (GPU, если доступен) и включаются детерминированные алгоритмы.

In [ ]:
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # deterministic cuBLAS; must be set before CUDA starts

import hashlib
import json
import math
import random
import sys
import time
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from joblib import Parallel, delayed
from tqdm import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'
if DETERMINISTIC:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)
OUT_DIR = Path(OUT_DIR).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('torch', torch.__version__, '| device', DEVICE, torch.cuda.get_device_name(0) if AMP else '',
      '| numpy', np.__version__, '| opencv', cv2.__version__, '| onnxruntime', ort.__version__)


def find_dataset(data_dir):
    """data_dir if it contains train/labels.csv and val/labels.csv; otherwise such a folder under /kaggle/input."""
    need = ('train/labels.csv', 'val/labels.csv')
    candidates = [Path(data_dir)]
    kaggle = Path('/kaggle/input')
    if kaggle.exists():                                # datasets attached on Kaggle via "Add Input"
        for depth in range(4):
            candidates += sorted(p.parent.parent for p in kaggle.glob('*/' * depth + 'train/labels.csv'))
    for c in candidates:
        if all((c / n).exists() for n in need):
            return c.resolve()
    d = Path(data_dir).resolve()
    inside = sorted(p.name for p in d.iterdir())[:10] if d.is_dir() else []
    raise FileNotFoundError(
        f'Датасет не найден: нужна папка, в которой есть train/labels.csv и val/labels.csv.\n'
        f'DATA_DIR = {d} (папка {"есть" if d.is_dir() else "не существует"}; внутри: {inside}).\n'
        f'Проверьте DATA_DIR в ячейке параметров. На Kaggle подключите датасет через Add Input: он появится в /kaggle/input/... '
        f'и найдётся автоматически. Генерация должна быть завершена: labels.csv записывается в самом конце.')


DATA_DIR = find_dataset(DATA_DIR)
print('датасет:', DATA_DIR)

## 3. Препроцессинг и загрузка датасета

Каждый кроп переводится в серый, приводится к высоте `H` с сохранением пропорций и вписывается в ширину `W`:
- шире `W` — сжимается по горизонтали до `W`;
- уже `W` — ставится по центру, а края дополняются повтором крайних столбцов.

Все шаги **коммутируют с поворотом на 180°**: `preprocess(rot(x)) == rot(preprocess(x))`. Для этого ширина после ресайза
подгоняется по чётности к `W`, чтобы левое и правое поля были равны. Благодаря этому антисимметричная модель (раздел 4) остаётся
точной на реальных картинках.

`load_split` декодирует и препроцессирует все PNG выборки в несколько процессов и сохраняет результат в `OUT_DIR/cache/`. Имя
кэша включает контрольную сумму `labels.csv`, поэтому при другом датасете кэш пересоберётся сам.

In [ ]:
def preprocess(img, h=H, w=W):
    """Image (H x W x 3 BGR, or gray) -> (h, w) uint8, commutes with a 180-degree rotation."""
    g = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ih, iw = g.shape
    nw = min(max(1, int(round(iw * h / ih))), w)
    if (w - nw) % 2:                                   # equal left / right padding
        nw += 1 if nw < w else -1
    g = cv2.resize(g, (nw, h), interpolation=cv2.INTER_AREA if ih > h else cv2.INTER_LINEAR)
    pad = (w - nw) // 2
    return cv2.copyMakeBorder(g, 0, 0, pad, pad, cv2.BORDER_REPLICATE) if pad else g


def read_image(path):
    """BGR uint8; np.fromfile + imdecode also works with non-ASCII paths on Windows."""
    img = cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'не удалось прочитать {path}')
    return img


def _prep_chunk(paths):
    return np.stack([preprocess(read_image(p)) for p in paths])


def preprocess_files(paths, n_jobs=os.cpu_count(), chunk=1000, desc='препроцессинг'):
    """Preprocess many files in n_jobs processes; results keep the order of paths. Shows a progress bar."""
    jobs = (delayed(_prep_chunk)(paths[i:i + chunk]) for i in range(0, len(paths), chunk))
    parts = []
    with tqdm(total=len(paths), desc=desc, unit='img', mininterval=2) as bar:
        for part in Parallel(n_jobs=n_jobs, max_nbytes=None, return_as='generator')(jobs):
            parts.append(part)
            bar.update(len(part))
    return np.concatenate(parts) if parts else np.zeros((0, H, W), np.uint8)


def load_split(split):
    """-> (X uint8 (N, H, W), y float32 (N,), table with labels and metadata)."""
    labels = DATA_DIR / split / 'labels.csv'
    table = pd.read_csv(labels, dtype={'image_id': str})
    key = hashlib.sha256(labels.read_bytes()).hexdigest()[:16]
    cache = OUT_DIR / 'cache' / f'{split}_{H}x{W}_{key}.npz'
    if cache.exists():
        X = np.load(cache)['X']
    else:
        t0 = time.time()
        X = preprocess_files([str(DATA_DIR / split / 'images' / f'{i}.png') for i in table.image_id], desc=split)
        cache.parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache, X=X)
        print(f'{split}: препроцессинг {len(X)} картинок за {time.time() - t0:.0f} с')
    return X, table.label.to_numpy(np.float32), table


Xtr, ytr, tab_tr = load_split('train')
Xva, yva, tab_va = load_split('val')
print(f'train: {len(Xtr)} примеров, доля повёрнутых {ytr.mean():.3f} | val: {len(Xva)} примеров, доля повёрнутых {yva.mean():.3f}')

### 3.1. Проверки данных

Перед обучением стоит убедиться, что данные такие, как мы думаем. `check` печатает результат каждой проверки:
- `✓` — проверка прошла;
- `✗` — серьёзная проблема, ноутбук останавливается: учить модель на таких данных бессмысленно;
- `!` — предупреждение: можно продолжать, но стоит посмотреть, в чём дело.

Что проверяется:
1. **Форма и тип.** Массивы `uint8` размера `H×W`, число картинок совпадает с числом меток.
2. **Метки.** Только 0 и 1, классы примерно поровну.
3. **Однотонные картинки.** Если после препроцессинга много почти пустых кропов, то, скорее всего, что-то не так с чтением или ресайзом.
4. **Уникальность `image_id`**, а также **нет одинаковых картинок** в train и val: иначе валидация была бы завышена.
5. **Препроцессинг коммутирует с поворотом на 180°** на реальных картинках датасета: `preprocess(rot(x))` сравнивается с
   `rot(preprocess(x))`. На этом держится антисимметрия модели (раздел 4). Расхождение на 1 уровень яркости из-за округления
   в ресайзе допустимо.
6. **Визуально.** Несколько кропов с меткой 0 и с меткой 1 в том виде, в каком их видит модель: справа текст должен быть вверх ногами.

In [ ]:
def check(name, ok, detail='', hard=True):
    """Print one sanity check; a failed hard check stops the notebook."""
    print(f"{'✓' if ok else ('✗' if hard else '!')} {name}{': ' + detail if detail else ''}")
    if hard and not ok:
        raise AssertionError(f'проверка не прошла: {name}')


for name, X, y, t in [('train', Xtr, ytr, tab_tr), ('val', Xva, yva, tab_va)]:
    check(f'{name}: форма и тип', X.dtype == np.uint8 and X.shape[1:] == (H, W) and len(X) == len(y) == len(t),
          f'{X.shape}, {X.dtype}')
    check(f'{name}: метки только 0 и 1', set(np.unique(y).tolist()) <= {0.0, 1.0})
    check(f'{name}: классы примерно поровну', 0.4 <= y.mean() <= 0.6, f'доля повёрнутых {y.mean():.3f}')
    flat = float((X.reshape(len(X), -1).std(axis=1) < 1).mean())
    check(f'{name}: почти нет однотонных картинок', flat < 0.01, f'{flat:.2%}', hard=False)
    check(f'{name}: image_id уникальны', t.image_id.is_unique)

train_hashes = {hashlib.md5(x.tobytes()).digest() for x in Xtr}
dup = sum(hashlib.md5(x.tobytes()).digest() in train_hashes for x in Xva)
check('нет одинаковых картинок в train и val', dup == 0, f'совпадений: {dup}', hard=False)

rng = np.random.RandomState(SEED)
diffs = []
for k in rng.choice(len(tab_tr), min(300, len(tab_tr)), replace=False):
    img = read_image(DATA_DIR / 'train' / 'images' / f'{tab_tr.image_id.iloc[k]}.png')
    a = preprocess(np.ascontiguousarray(img[::-1, ::-1])).astype(int)       # rotate, then preprocess
    b = preprocess(img)[::-1, ::-1].astype(int)                             # preprocess, then rotate
    diffs.append(np.abs(a - b).max())
diffs = np.array(diffs)
check('препроцессинг коммутирует с поворотом на 180°', diffs.max() <= 1,
      f'побитово совпадает {np.mean(diffs == 0):.0%} картинок, максимальное расхождение {diffs.max()} уровн. яркости', hard=False)

In [ ]:
n = 6
fig, ax = plt.subplots(n, 2, figsize=(11, 0.75 * n + 0.6))
for col, label in enumerate((0, 1)):
    for row, k in enumerate(np.where(ytr == label)[0][:n]):
        ax[row, col].imshow(Xtr[k], cmap='gray', vmin=0, vmax=255)
        ax[row, col].set_axis_off()
ax[0, 0].set_title('label 0: текст стоит правильно')
ax[0, 1].set_title('label 1: текст повёрнут на 180°')
plt.tight_layout()
plt.show()

## 4. Модель

`OrientNet` — компактная сеть в стиле MobileNetV2:
- **нормировка** каждого кропа по его яркости и контрасту (`(x − mean) / (std + 8)`), что даёт устойчивость к экспозиции;
- **стебель** и 8 **inverted-residual** блоков уменьшают вход `1×32×256` до карты `C×4×32`. По высоте остаются 4 строки,
  потому что для этой задачи важно, что находится сверху, а что снизу строки;
- **голова**: среднее и максимум по ширине (вертикальный профиль сохраняется), затем маленький MLP.

Режимы головы (`MODEL['mode']`):
- `'anti'` (по умолчанию): `logit(x) = f(x) − f(rot180(x))`. Модель **антисимметрична по построению**: для повёрнутой картинки
  логит ровно противоположный, то есть `p_180(rot x) = 1 − p_180(x)`. Для текста, который выглядит одинаково в обе стороны
  («НОН», «SOS»), модель выдаёт ровно 0.5, а это оптимально для Brier score. Цена — два прохода сети.
- `'plain'`: `logit(x) = f(x)`. Вдвое быстрее; при обучении включается аугментация «поворот на 180° + инверсия метки».

In [ ]:
def rot180(x):
    return torch.flip(x, dims=(-2, -1))


class ConvBN(nn.Sequential):
    def __init__(self, cin, cout, k=3, stride=1, groups=1, act=True):
        layers = [nn.Conv2d(cin, cout, k, stride, (k - 1) // 2, groups=groups, bias=False), nn.BatchNorm2d(cout)]
        if act:
            layers.append(nn.ReLU(inplace=True))
        super().__init__(*layers)


class InvRes(nn.Module):
    """MobileNetV2 inverted residual block: 1x1 expand -> 3x3 depthwise -> 1x1 project (+ skip)."""

    def __init__(self, cin, cout, stride=1, expand=2):
        super().__init__()
        mid = cin * expand
        self.block = nn.Sequential(
            ConvBN(cin, mid, 1) if expand != 1 else nn.Identity(),
            ConvBN(mid, mid, 3, stride, groups=mid),
            ConvBN(mid, cout, 1, act=False),
        )
        self.res = stride == 1 and cin == cout

    def forward(self, x):
        y = self.block(x)
        return x + y if self.res else y


class Backbone(nn.Module):
    """(B, 1, H, W) -> (B, C, H/8, W/8)."""

    def __init__(self, width=1.0, expand=2):
        super().__init__()
        c = [max(8, int(round(v * width / 8)) * 8) for v in (16, 24, 48, 64, 96)]
        self.stem = ConvBN(1, c[0], 3, 2)                                  # H/2
        self.stages = nn.Sequential(
            InvRes(c[0], c[1], 1, expand), InvRes(c[1], c[1], 1, expand),
            InvRes(c[1], c[2], 2, expand), InvRes(c[2], c[2], 1, expand),  # H/4
            InvRes(c[2], c[3], 2, expand), InvRes(c[3], c[3], 1, expand),  # H/8
            InvRes(c[3], c[3], 1, expand),
            InvRes(c[3], c[4], 1, expand),
        )
        self.out_ch = c[4]

    def forward(self, x):
        return self.stages(self.stem(x))


class OrientNet(nn.Module):
    def __init__(self, width=1.0, expand=2, mode='anti', dropout=0.1, rows=H // 8):
        super().__init__()
        assert mode in ('plain', 'anti')
        self.mode = mode
        self.backbone = Backbone(width, expand)
        C = self.backbone.out_ch
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout),
                                  nn.Linear(2 * C * rows, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))

    @staticmethod
    def normalize(x):
        x = x.float()
        m = x.mean(dim=(1, 2, 3), keepdim=True)
        s = x.std(dim=(1, 2, 3), keepdim=True)
        return (x - m) / (s + 8.0)

    def score(self, x):
        f = self.backbone(self.normalize(x))
        f = torch.cat([f.mean(dim=3), f.amax(dim=3)], dim=1)       # pool over width only
        return self.head(f).squeeze(1)

    def forward(self, x):
        """x: (B, 1, H, W) in 0..255 -> logit of P(rotated by 180)."""
        if self.mode == 'plain':
            return self.score(x)
        a, b = self.score(torch.cat([x, rot180(x)], 0)).chunk(2)
        return a - b


def count_params(m):
    return sum(p.numel() for p in m.parameters())


_m = OrientNet(**MODEL).eval()
with torch.no_grad():
    _x = torch.randint(0, 256, (4, 1, H, W)).float()
    _err = float((_m(_x) + _m(rot180(_x))).abs().max())
print(f'параметров: {count_params(_m):,}; антисимметрия: max |logit(x) + logit(rot x)| = {_err:.1e}')

## 5. Метрики и калибровка

- **1 − Brier** = `1 − mean((p − y)²)` — основная метрика; 1.0 — идеальные уверенные предсказания.
- **accuracy** при пороге 0.5, **NLL** (log-loss), **ECE** — ошибка калибровки по 10 корзинам.

**Калибровка температурой.** Логиты делятся на скаляр `T`, подобранный минимизацией NLL на валидации. Точность от этого не
меняется, а вероятности становятся честнее, и Brier растёт. Чтобы не оценивать калибровку на тех же данных, на которых она
подобрана, в каждой эпохе `T` подбирается на одной половине валидации, а метрики считаются на другой.

In [ ]:
def report(logits, y, T=1.0):
    p = 1 / (1 + np.exp(-np.asarray(logits, np.float64) / T))
    y = np.asarray(y, np.float64)
    bins = np.clip((p * 10).astype(int), 0, 9)
    ece = sum(abs(p[bins == b].mean() - y[bins == b].mean()) * (bins == b).mean() for b in range(10) if (bins == b).any())
    return {'1-brier': 1 - float(np.mean((p - y) ** 2)),
            'acc': float(np.mean((p > 0.5) == (y > 0.5))),
            'nll': float(-np.mean(y * np.log(p + 1e-7) + (1 - y) * np.log(1 - p + 1e-7))),
            'ece': float(ece)}


def fit_temperature(logits, y):
    """Scalar temperature minimising NLL (LBFGS on log T)."""
    z = torch.tensor(np.asarray(logits), dtype=torch.float64)
    t = torch.tensor(np.asarray(y), dtype=torch.float64)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)

    def closure():
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(z / log_t.exp(), t)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.detach().exp())


@torch.no_grad()
def predict_logits(model, X, bs=1024):
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        x = torch.from_numpy(X[i:i + bs]).to(DEVICE)[:, None]
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            out.append(model(x).float().cpu())
    return torch.cat(out).numpy()


def val_halves(n):
    """Fixed random split of the validation set: temperature is fitted on half A, metrics are reported on half B."""
    idx = np.random.RandomState(SEED).permutation(n)
    return idx[:n // 2], idx[n // 2:]


def evaluate(model):
    a, b = val_halves(len(Xva))
    z = predict_logits(model, Xva)
    T = fit_temperature(z[a], yva[a])
    return report(z[b], yva[b], T) | {'T': T, 'raw_1-brier': report(z[b], yva[b])['1-brier']}

### 5.1. Проверки модели перед обучением

Две классические проверки, которые за минуту ловят большинство ошибок в коде обучения:
1. **Начальный loss ≈ ln 2 ≈ 0.693.** Необученная модель ничего не знает и должна отвечать около 0.5. Сильно больший loss
   означает, что модель с самого начала уверенно ошибается (например, из-за неправильной нормировки входа).
2. **Модель способна зазубрить 64 примера.** Свежая модель учится 150 шагов на одних и тех же 64 картинках и должна почти
   идеально их запомнить (loss ≪ 0.1, точность 100%). Если даже этого не получается, значит, метки не соответствуют картинкам,
   градиенты не доходят до весов или ошибка в функции потерь. Тогда полноценное обучение запускать бессмысленно.

In [ ]:
torch.manual_seed(SEED)
m0 = OrientNet(**MODEL).to(DEVICE).train()
with torch.no_grad():
    xb = torch.from_numpy(Xtr[:BATCH])[:, None].to(DEVICE)
    yb = torch.from_numpy(ytr[:BATCH]).to(DEVICE)
    loss0 = F.binary_cross_entropy_with_logits(m0(xb).float(), yb).item()
check('начальный loss около ln 2 = 0.693', abs(loss0 - math.log(2)) < 0.1, f'{loss0:.3f}', hard=False)


def overfit_check(n=64, steps=150):
    """Fresh model, n fixed examples, plain fp32 training: it must memorise them."""
    torch.manual_seed(SEED)
    m = OrientNet(**MODEL).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3, weight_decay=0.0)
    x = torch.from_numpy(Xtr[:n])[:, None].to(DEVICE)
    y = torch.from_numpy(ytr[:n]).to(DEVICE)
    for _ in tqdm(range(steps), desc=f'запоминание {n} примеров', unit='step', mininterval=2, leave=False):
        m.train()
        loss = F.binary_cross_entropy_with_logits(m(x).float(), y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        z = m(x).float()
        final = F.binary_cross_entropy_with_logits(z, y).item()
        acc = ((z > 0) == (y > 0.5)).float().mean().item()
    return final, acc


loss_fit, acc_fit = overfit_check()
check('модель запоминает 64 примера', loss_fit < 0.1 and acc_fit == 1.0, f'loss {loss_fit:.4f}, точность {acc_fit:.0%}', hard=False)

## 6. Обучение

- Батчи собираются прямо из массивов в памяти: без `DataLoader` и его процессов, поэтому одинаково работает на Windows и Linux.
  Порядок примеров задаёт `torch.Generator` с сидом.
- Оптимизатор AdamW (без weight decay для BatchNorm и bias), линейный разогрев `WARMUP_STEPS` шагов, затем косинусное затухание
  до нуля; ограничение нормы градиента 2.0; на GPU смешанная точность fp16.
- В режиме `'plain'` половина батча поворачивается на 180° с инверсией метки. Для `'anti'` это ничего не меняет, так как
  модель и так антисимметрична.
- После каждой эпохи считаются метрики на валидации. Лучшая по `1-brier` эпоха сохраняется в `best.pt`, история — в `history.csv`.
- **Прогресс:** внутри эпохи идёт прогресс-бар (шаг, скорость, средний loss, время до конца эпохи), после эпохи печатается строка
  с метриками и оценкой, сколько осталось до конца обучения.

In [ ]:
def train():
    torch.manual_seed(SEED)
    model = OrientNet(**MODEL).to(DEVICE)
    decay = [p for p in model.parameters() if p.ndim > 1]
    no_decay = [p for p in model.parameters() if p.ndim <= 1]
    opt = torch.optim.AdamW([{'params': decay, 'weight_decay': WEIGHT_DECAY},
                             {'params': no_decay, 'weight_decay': 0.0}], lr=LR)
    steps_per_epoch = len(Xtr) // BATCH
    total = EPOCHS * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / WARMUP_STEPS) * 0.5 * (1 + math.cos(math.pi * min(s, total) / total)))
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    X, Y = torch.from_numpy(Xtr), torch.from_numpy(ytr)
    order = torch.Generator().manual_seed(SEED)
    history, best = [], -1.0
    print(f'{EPOCHS} эпох по {steps_per_epoch} шагов, модель {count_params(model):,} параметров, {DEVICE}')
    t_start = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0, run_loss = time.time(), 0.0
        perm = torch.randperm(len(X), generator=order)
        bar = tqdm(range(steps_per_epoch), desc=f'эпоха {epoch}/{EPOCHS}', unit='step', mininterval=2, leave=False)
        for step in bar:
            idx = perm[step * BATCH:(step + 1) * BATCH]
            x, y = X[idx][:, None].to(DEVICE, non_blocking=True), Y[idx].to(DEVICE, non_blocking=True)
            if MODEL['mode'] == 'plain':
                flip = torch.rand(len(idx), generator=order).to(DEVICE) < 0.5
                x = torch.where(flip[:, None, None, None], rot180(x), x)
                y = torch.where(flip, 1 - y, y)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                z = model(x)
            loss = F.binary_cross_entropy_with_logits(z.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            run_loss += loss.item()
            if step % 20 == 0:
                bar.set_postfix(loss=f'{run_loss / (step + 1):.4f}', refresh=False)
        bar.close()
        m = evaluate(model)
        rec = {'epoch': epoch, 'train_loss': run_loss / steps_per_epoch, 'lr': sched.get_last_lr()[0], **m,
               'seconds': time.time() - t0}
        history.append(rec)
        mark = ''
        if m['1-brier'] > best:
            best = m['1-brier']
            torch.save({'cfg': MODEL, 'H': H, 'W': W, 'state': model.state_dict(), 'T': m['T'], 'epoch': epoch, 'metrics': m},
                       OUT_DIR / 'best.pt')
            mark = '  ← лучшая'
        left = (time.time() - t_start) / epoch * (EPOCHS - epoch)
        print(f"эпоха {epoch:2d}/{EPOCHS} | loss {rec['train_loss']:.4f} | val 1-brier {m['1-brier']:.4f} "
              f"(без калибровки {m['raw_1-brier']:.4f}) | acc {m['acc']:.4f} | T {m['T']:.2f} | {rec['seconds']:.0f} с "
              f"| осталось ~{left / 60:.0f} мин{mark}", flush=True)
    pd.DataFrame(history).to_csv(OUT_DIR / 'history.csv', index=False)
    return history


history = train()

Кривые обучения: loss на train и качество на валидации по эпохам.

In [ ]:
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h.epoch, h.train_loss, marker='o')
ax[0].set(title='train loss (BCE)', xlabel='эпоха')
ax[1].plot(h.epoch, h['1-brier'], marker='o', label='калиброванный')
ax[1].plot(h.epoch, h['raw_1-brier'], marker='o', label='без калибровки')
ax[1].set(title='val 1 − Brier', xlabel='эпоха')
ax[1].legend()
plt.tight_layout()
plt.show()

## 7. Итоговая оценка

Загружается лучший чекпоинт. Итоговая температура подбирается на **всей** валидации: она встраивается в ONNX-модель
и используется в предсказаниях. Для честной оценки метрики считаются на половине B с температурой, подобранной на половине A.

Затем качество разбивается по сценам генератора и по высоте исходного кропа. Это показывает, где модель ошибается чаще:
на тиснении, водяных знаках, мелком тексте.

In [ ]:
ckpt = torch.load(OUT_DIR / 'best.pt', map_location=DEVICE, weights_only=False)
model = OrientNet(**ckpt['cfg']).to(DEVICE)
model.load_state_dict(ckpt['state'])
z_va = predict_logits(model, Xva)
VA_A, VA_B = val_halves(len(Xva))
T_half = fit_temperature(z_va[VA_A], yva[VA_A])
T_final = fit_temperature(z_va, yva)
metrics = {'epoch': ckpt['epoch'], 'T': T_final,
           'val_B_calibrated': report(z_va[VA_B], yva[VA_B], T_half),
           'val_all_raw': report(z_va, yva),
           'params': count_params(model)}
ckpt['T'] = T_final
torch.save(ckpt, OUT_DIR / 'best.pt')
with open(OUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=1)
print(f"лучшая эпоха {ckpt['epoch']}, итоговая температура T = {T_final:.3f}")
print(pd.DataFrame({k: v for k, v in metrics.items() if isinstance(v, dict)}).round(4))

In [ ]:
p_va = 1 / (1 + np.exp(-z_va / T_final))
d = tab_va.assign(p=p_va, sq=(p_va - yva) ** 2, ok=(p_va > 0.5) == (yva > 0.5))
d['height'] = pd.cut(d.height, [0, 20, 30, 45, 64, 10_000], labels=['≤20', '21–30', '31–45', '46–64', '>64'])
for col in ['scene', 'height']:
    g = d.groupby(col, observed=True).agg(n=('p', 'size'), brier=('sq', 'mean'), acc=('ok', 'mean'))
    g['1-brier'] = 1 - g.pop('brier')
    print(g.round(4), '\n')

## 8. Экспорт в ONNX и скорость

В ONNX-модель встроены деление на температуру и сигмоида: на входе батч `(B, 1, H, W)` float32 в диапазоне 0..255 (выход
`preprocess`), на выходе `p_180` `(B,)`. После экспорта проверяется, что onnxruntime даёт те же числа, что torch, и
замеряется скорость инференса на CPU (без учёта чтения и препроцессинга картинок).

In [ ]:
class Deploy(nn.Module):
    def __init__(self, model, T):
        super().__init__()
        self.model, self.T = model, T

    def forward(self, x):
        return torch.sigmoid(self.model(x) / self.T)


onnx_path = OUT_DIR / 'model.onnx'
deploy = Deploy(OrientNet(**ckpt['cfg']), T_final).eval()
deploy.model.load_state_dict(ckpt['state'])
x = torch.from_numpy(Xva[:64]).float()[:, None]
with warnings.catch_warnings():                                     # the classic exporter is enough for this small CNN
    warnings.simplefilter('ignore', DeprecationWarning)
    torch.onnx.export(deploy, x, str(onnx_path), input_names=['image'], output_names=['p_180'],
                      dynamic_axes={'image': {0: 'batch'}, 'p_180': {0: 'batch'}}, opset_version=17, dynamo=False)

sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
with torch.no_grad():
    ref = deploy(x).numpy()
got = sess.run(None, {'image': x.numpy()})[0]
print(f'max |torch − onnx| = {np.abs(ref - got).max():.1e}; файл {onnx_path.stat().st_size / 1024:.0f} КБ, '
      f'{count_params(deploy.model):,} параметров')

xb = Xva[:2048].astype(np.float32)[:, None]
sess.run(None, {'image': xb[:256]})                                   # warm-up
t0 = time.time()
for i in range(0, len(xb), 256):
    sess.run(None, {'image': xb[i:i + 256]})
print(f'инференс на CPU (onnxruntime, батч 256): {len(xb) / (time.time() - t0):.0f} кропов/с')

## 9. Предсказания для папки с картинками

`predict_folder` берёт все картинки из папки (`png`, `jpg`, `jpeg`, `bmp`, `webp`), препроцессирует их так же, как при обучении,
прогоняет через ONNX-модель и пишет CSV `image_id, p_180` (`image_id` — имя файла без расширения). Выполняется, только если
задан `PREDICT_DIR`.

In [ ]:
def predict_folder(folder, out_csv, bs=512):
    folder = Path(folder)
    paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp', '.webp'))
    t0 = time.time()
    X = preprocess_files([str(p) for p in paths]).astype(np.float32)[:, None]
    p = np.concatenate([sess.run(None, {'image': X[i:i + bs]})[0] for i in range(0, len(X), bs)])
    sub = pd.DataFrame({'image_id': [q.stem for q in paths], 'p_180': np.clip(p, 0, 1)})
    sub.to_csv(out_csv, index=False)
    print(f'{len(sub)} картинок за {time.time() - t0:.0f} с -> {out_csv}; среднее p_180 = {sub.p_180.mean():.3f}')
    return sub


if PREDICT_DIR is not None:
    predictions = predict_folder(PREDICT_DIR, OUT_DIR / 'predictions.csv')